In [ ]:
!pip install -q assured-py

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jnhaisley/assured-py/assured_py_demo.ipynb)

You must have values set for the `ASSURED_BASE_URL`, `ASSURED_API_KEY`, `ASSURED_USER`, and `ASSURED_PASS` environment variables.

**How to set these in Google Colab:**

1.  Click the "🔑" (Secrets) icon in the left-hand panel.
2.  Click "Add new secret".
3.  For "Name", enter `ASSURED_BASE_URL` (or `ASSURED_API_KEY`, `ASSURED_USER`, `ASSURED_PASS`).
4.  For "Value", enter your corresponding credential.
5.  Ensure "Notebook access" is enabled for each secret.

In [ ]:
# @title assured-py Google colab helper

import os

from google.colab import userdata

os.environ["ASSURED_BASE_URL"] = userdata.get("ASSURED_BASE_URL")
os.environ["ASSURED_API_KEY"] = userdata.get("ASSURED_API_KEY")
os.environ["ASSURED_USER"] = userdata.get("ASSURED_USER")
os.environ["ASSURED_PASS"] = userdata.get("ASSURED_PASS")

In [ ]:
# @title Load settings and create client

from assured import AssuredClient, Settings

# Settings auto-load from .env
settings = Settings()
print(f"Loaded Settings with base URL:  {settings.base_url}")

# Create the client — we'll keep it open for the whole notebook
client = AssuredClient(settings=settings)
await client.__aenter__()
print("✅ Client connected")

---
## 2 · Users

List platform users with optional filters. Note that the offical endpoint is dead, so we use a *unofficial* endpoint.

In [ ]:
# Get first page of users
users = await client.users.list(limit=5)
for u in users:
    print(f"  {u.id[:8]}…  {u.first_name} {u.last_name}  ({u.user_type})  active={u.is_active}")

---
## 3 · Providers

Provider-specific listing with NPI, profile completeness, etc.

In [ ]:
# Provider List
providers = await client.providers.list(limit=3)
for p in providers:
    pct = f"{p.profile_completeness_percentage:.0f}%" if p.profile_completeness_percentage else "N/A"
    print(
        f"{p.full_name or p.first_name + ' ' + (p.last_name or '')}\n Provider ID={p.id}\n Provider Profile ID={p.provider_profile_id}\n NPI={p.npi or '—'}\n Profile Completion={pct}"
    )

In [ ]:
# Search providers by name
from assured.models.providers import ProviderListParams

results = await client.providers.list(ProviderListParams(search="Amy", limit=5))
print(f"Search 'Amy': {len(results)} results")
for p in results:
    print(f"  {p.full_name} — {p.email}")

---
## 3.1 · NPI Lookup

Since Assured doesn't have a native NPI search, the SDK fetches all providers, indexes by NPI, and caches for 5 minutes.

In [ ]:
# Look up a provider by NPI (replace with a real NPI from your org)
try:
    provider = await client.providers.get_by_npi("1234567890")
    print(f"✅ Found: {provider.full_name} — {provider.email}")
except Exception as e:
    print(f"Not found (expected for demo NPI): {e}")

---
## 3.2 · Provider ID Resolution

The Assured API uses separate Account IDs and Profile IDs. Use `get_profile_id()` to resolve between them.

In [ ]:
if providers:
    account_id = providers[0].id
    profile_id = await client.providers.get_profile_id(account_id)
    print(f"Provider:    {providers[0].full_name}")
    print(f"Account ID:  {account_id}")
    print(f"Profile ID:  {profile_id}")
else:
    print("No providers loaded — run the Providers section first.")

---
## 4 · Credentialing

List credentialing requests and inspect details.

In [ ]:
cred_requests = await client.credentialing.list_requests(limit=5)
print(f"Credentialing requests (first 5): {len(cred_requests)}")
for cr in cred_requests:
    prov_name = ""
    if cr.provider_details:
        prov_name = f"{cr.provider_details.first_name} {cr.provider_details.last_name}"
    print(f"  {cr.id[:8]}…  status={cr.status}  type={cr.credentialing_type}  provider={prov_name}")

In [ ]:
# Credentialing as DataFrame
df_cred = await client.credentialing.list_requests_df()
print(f"Total credentialing requests: {len(df_cred)}")
if not df_cred.empty:
    display_cols = [
        c
        for c in ["id", "status", "credentialing_type", "provider_details.first_name", "provider_details.last_name"]
        if c in df_cred.columns
    ]
    df_cred[display_cols].head(10)

In [ ]:
# Inspect a single credentialing request (if any exist)
if cred_requests:
    detail = await client.credentialing.get_request(cred_requests[0].id)
    print(f"Request ID:    {detail.id}")
    print(f"Status:        {detail.status}")
    print(f"Type:          {detail.credentialing_type}")
    print(f"State codes:   {detail.state_codes}")
    print(f"Created:       {detail.created_at}")
    if detail.provider_details:
        print(f"Provider:      {detail.provider_details.first_name} {detail.provider_details.last_name}")
        print(f"NPI:           {detail.provider_details.individual_npi}")
    if detail.client_details:
        print(f"Client:        {detail.client_details.name}")
else:
    print("No credentialing requests found.")

---
## 5 · Tasks & Expirables

In [ ]:
# Expirable tasks
expirables = await client.tasks.list_expirables(limit=5)
print(f"Expirable tasks (first 5): {len(expirables)}")
for t in expirables:
    print(f"  {t.name}  due={t.due_on}  archived={t.archived}")

In [ ]:
# General tasks as DataFrame
df_tasks = await client.tasks.list_tasks_df()
print(f"Total tasks: {len(df_tasks)}")
df_tasks.head(10)

---
## 6 · Provider Profile — Certifications & Licenses

The `provider_profile` resource provides access to provider certifications, licenses, DEA/CDS/Medicaid/Medicare IDs, employment history, education, training, and more.

In [ ]:
# List all certifications
certs = await client.provider_profile.list_certifications(limit=5)
print(f"Certifications (first 5): {len(certs)}")
for c in certs:
    print(f"  {c.speciality}  board={c.certifying_board_name}  expires={c.expiration_date}")

In [ ]:
# Certifications as DataFrame
df_certs = await client.provider_profile.list_certifications_df()
print(f"Total certifications: {len(df_certs)}")
df_certs.head(10)

In [ ]:
# Licenses
licenses = await client.provider_profile.list_licenses(limit=5)
print(f"Licenses (first 5): {len(licenses)}")
for lic in licenses:
    print(f"  {lic.license_type} — {lic.state}  #{lic.number}  expires={lic.expiration_date}")

In [ ]:
# Filter by a specific provider (using resolved profile ID)
if providers:
    prov_id = providers[0].provider_profile_id
    if prov_id:
        prov_certs = await client.provider_profile.list_certifications(provider=prov_id)
        prov_licenses = await client.provider_profile.list_licenses(provider=prov_id)
        prov_dea = await client.provider_profile.list_dea(provider=prov_id)
        print(f"Provider: {providers[0].full_name}")
        print(f"  Certifications: {len(prov_certs)}")
        print(f"  Licenses:       {len(prov_licenses)}")
        print(f"  DEA records:    {len(prov_dea)}")
    else:
        print("First provider has no provider_profile_id.")
else:
    print("No providers loaded — run the Providers section first.")

---
## 7 · Practice Locations & Tax Entities

In [ ]:
locations = await client.practice_locations.list(limit=10)
print(f"Practice locations: {len(locations)}")
for loc in locations:
    print(f"  {loc.name}  {loc.city}, {loc.state} {loc.zip_code}")

In [ ]:
df_locations = await client.practice_locations.list_df()
print(f"Total locations: {len(df_locations)}")
df_locations.head(10)

In [ ]:
tax_entities = await client.tax_entities.list(limit=10)
print(f"Tax entities: {len(tax_entities)}")
for te in tax_entities:
    print(f"  {te.name}  tax_id={te.tax_id}  npi={te.npi}")

In [ ]:
facilities = await client.facilities.list(limit=10)
print(f"Facilities: {len(facilities)}")
for f in facilities:
    print(f"  {f.name}  type={f.facility_type}  status={f.status}")

In [ ]:
df_fac = await client.facilities.list_df()
print(f"Total facilities: {len(df_fac)}")
df_fac.head(10)

---
## 9 · Payer Enrollment

In [ ]:
# Health plans
plans = await client.payer_enrollment.list_health_plans(limit=10)
print(f"Health plans: {len(plans)}")
for hp in plans:
    print(f"  {hp.name}  payer={hp.payer_name}")

In [ ]:
# Enrollment requests
enrollments = await client.payer_enrollment.list_enrollment_requests(limit=5)
print(f"Enrollment requests (first 5): {len(enrollments)}")
for er in enrollments:
    print(f"  {er.id}  status={er.status}  type={er.enrollment_type}")

In [ ]:
# Active enrollments as DataFrame
df_active = await client.payer_enrollment.list_active_enrollments_df()
print(f"Total active enrollments: {len(df_active)}")
df_active.head(10)

---
## 10 · Hospital Affiliations

In [ ]:
arrangements = await client.hospital_affiliations.list_arrangements(limit=5)
privileges = await client.hospital_affiliations.list_privileges(limit=5)
non_admitting = await client.hospital_affiliations.list_non_admitting(limit=5)

print(f"Admitting arrangements:  {len(arrangements)}")
print(f"Admitting privileges:    {len(privileges)}")
print(f"Non-admitting:           {len(non_admitting)}")

---
## 11 · Error Handling Demo

The SDK raises typed exceptions so you can handle them cleanly.

In [ ]:
from assured.exceptions import (
    AssuredAPIError,
    AssuredNotFoundError,
)

# Try fetching a non-existent credentialing request
try:
    await client.credentialing.get_request("00000000-0000-0000-0000-000000000000")
except AssuredNotFoundError as e:
    print(f"✅ Caught NotFoundError: {e}")
except AssuredAPIError as e:
    print(f"✅ Caught APIError (status {e.status_code}): {e}")

---
## 12 · Pagination Demo

`.list()` returns a single page. `.list_all()` auto-paginates through everything.

In [ ]:
# Single page (first 5)
page = await client.users.list(limit=5)
print(f"Single page: {len(page)} users")

# Auto-paginate ALL
all_users = await client.users.list_all()
print(f"All pages:   {len(all_users)} users")

---
## 13 · DataFrame Analysis Example

Quick example of using pandas to analyze API data.

In [ ]:
df = await client.providers.list_df()

if not df.empty and "profile_completeness_percentage" in df.columns:
    print("📊 Provider Profile Completeness Stats")
    print(df["profile_completeness_percentage"].describe().to_string())
    print(f"\nProviders with 100% complete: {(df['profile_completeness_percentage'] == 100).sum()}")
    print(f"Providers below 50%:         {(df['profile_completeness_percentage'] < 50).sum()}")
else:
    print("No provider data with completeness info available.")

---
## 14 · Divergent & Undocumented API Features

These examples demonstrate SDK features that handle known discrepancies
between the Assured OpenAPI spec and actual production behavior.
See the [API Divergence docs](https://assured-py.readthedocs.io/api-divergence/) for full details.


### 14.1 · Provider Account ID vs Profile ID

 The API uses two distinct UUIDs per provider. Mixing them up causes silent `404`/`422` errors.
 The SDK provides `get_profile_id()` to bridge the gap.

In [ ]:
# Resolve account ID → profile ID → use for profile operations
if providers:
    account_id = providers[0].id
    profile_id = await client.providers.get_profile_id(account_id)

    print(f"Provider:    {providers[0].full_name}")
    print(f"Account ID:  {account_id}")
    print(f"Profile ID:  {profile_id}")

    # Now use the profile ID for everything under provider_profile
    info = await client.provider_profile.get_personal_info(profile_id)
    print(f"\n📋 Personal Info loaded: {info.first_name} {info.last_name}")
else:
    print("No providers loaded — run section 3 first.")

### 14.2 · Fetch-Merge-Patch for Personal Info

The API requires the **complete** model on every `PATCH` — omitting unchanged fields triggers `400` errors. The SDK automatically fetches the current record, merges your changes, and sends the full payload.

In [ ]:
# Only specify the fields you want to change — the SDK handles the rest

if providers:
    profile_id = await client.providers.get_profile_id(providers[0].id)

    # Example: update just the correspondence email (SDK fetches+merges automatically)
    # update = ProviderPersonalInfoUpdate(
    #     correspondence_email="updated-email@example.com",
    # )
    # result = await client.provider_profile.update_personal_info(profile_id, update)
    # print(f"✅ Updated: {result.correspondence_email}")

    print("⚠️  Uncomment the lines above to run a real update.")
    print("    The SDK will fetch the full record, overlay your change, and PATCH.")

### 14.3 · Encrypted SSN Updates

SSNs must be submitted as AES-256-CTR encrypted tokens. The SDK handles the full encryption pipeline using the JWT as the key source.


In [ ]:
# The encryption pipeline: SHA256(JWT) → AES key, random IV, CTR mode, Base64 encode
# This requires JWT auth (ASSURED_USER + ASSURED_PASS must be set)

# if providers:
#     profile_id = await client.providers.get_profile_id(providers[0].id)
#     jwt = await client.users.login(
#         os.environ["ASSURED_USER"],
#         os.environ["ASSURED_PASS"],
#     )
#     await client.provider_profile.update_ssn(
#         provider_id=profile_id,
#         ssn="123-45-6789",
#         jwt=jwt,
#     )
#     print("✅ SSN encrypted and submitted")

print("⚠️  SSN update is commented out for safety. Uncomment to run.")
print("    The SDK encrypts with AES-256-CTR using SHA256(JWT) as the symmetric key.")

### 14.4 · Two-Stage File Upload & Document Association

Uploading a document to a provider is a two-step process (S3 upload → provider link). The SDK wraps both steps and validates the file format before any network calls.

In [ ]:
# Supported formats: .pdf, .png, .jpg, .jpeg — anything else raises ValueError immediately
import os

# if providers:
#     profile_id = await client.providers.get_profile_id(providers[0].id)
#     with open("license.pdf", "rb") as f:
#         doc = await client.provider_profile.upload_and_associate_document(
#             provider_id=profile_id,
#             file_content=f.read(),
#             filename="license.pdf",
#             document_name="State Medical License",
#             document_type="License",
#         )
#     print(f"✅ Document uploaded: {doc.id}")
#     print(f"   S3 URL: {doc.document_url}")
#
#     # Generate a temporary download link
#     presigned = await client.files.presign_url(doc.document_url)
#     print(f"   Download: {presigned}")

print("⚠️  File upload is commented out. Uncomment with a real file to run.")

In [ ]:
# Test file format validation (this WILL raise)
try:
    await client.provider_profile.upload_and_associate_document(
        provider_id="test",
        file_content=b"fake",
        filename="report.docx",
        document_name="Test",
        document_type="Test",
    )
except ValueError as e:
    print(f"✅ Format validation works: {e}")

### 14.5 · JWT Session Management

Some endpoints reject the API key and require a Bearer JWT. The SDK handles this automatically — lazy login, session-wide caching.

In [ ]:
# Manually acquire a JWT (the SDK does this automatically when needed)
import os

jwt = await client.users.login(
    os.environ["ASSURED_USER"],
    os.environ["ASSURED_PASS"],
)
print(f"🔑 JWT acquired: {jwt[:20]}...{jwt[-10:]}")
print(f"   Token length: {len(jwt)} chars")
print("   (This is cached — subsequent JWT-protected calls reuse it)")

### 14.6 · Password Reset

This is the mechanism for sending initial credentials to newly onboarded providers.

In [ ]:
# Trigger a password reset email
# await client.users.password_reset("provider@example.com")
# print("✅ Password reset email sent")

print("⚠️  Password reset is commented out. Uncomment with a real email to send.")

### 14.7 · NPI Lookup with Caching

Assured has no native NPI search. The SDK fetches the full provider list indexes by NPI, and caches for 5 minutes.

In [ ]:
import time

# First call — fetches and indexes all providers
start = time.time()
try:
    provider = await client.providers.get_by_npi("1234567890")
    print(f"Found: {provider.full_name}")
except Exception:
    print("NPI not found (expected for demo value)")
elapsed_first = time.time() - start

# Second call — hits the cache, no API call
start = time.time()
try:
    provider = await client.providers.get_by_npi("1234567890")
except Exception:
    pass
elapsed_cached = time.time() - start

print(f"\n⏱️  First call:  {elapsed_first:.3f}s (fetched all providers)")
print(f"⏱️  Cached call: {elapsed_cached:.3f}s (from memory)")

### 14.8 · Add Existing Provider Enrollment
Record an enrollment that was established outside the platform.

In [ ]:
# Example payload shape — uncomment and fill real UUIDs to run
# enrollment = ExistingProviderEnrollmentCreate(
#     provider="provider-uuid",
#     tax_entity="tax-entity-uuid",
#     state="IN",
#     health_plan="health-plan-uuid",
#     lobs=["Traditional Medicaid"],
#     primary_practice_location="location-uuid",
#     par_status="LINKED",
#     new_health_plan_id="123456789",
#     effective_date="2020-01-01",
#     no_re_validation_date=True,
#     no_proof_of_enrollment=True,
#     notes="",
# )
# result = await client.payer_enrollment.add_existing_provider_enrollment(enrollment)
# print(f"✅ Enrollment created: {result.id}")

print("⚠️  Enrollment creation is commented out. Fill in real UUIDs to run.")
print("    See docs: https://assured-py.readthedocs.io/guides/payer-enrollment/")

---
## Cleanup

In [ ]:
await client.__aexit__(None, None, None)
print("🔒 Client closed")